In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:

url = "https://raw.githubusercontent.com/gildasedenakpo-sketch/Data_set_home/refs/heads/main/GoogleAds_DataAnalytics_Sales_Uncleaned.csv"

In [ ]:
df = pd.read_csv(url)

In [ ]:
df.head() # to show the head of my dataset

In [ ]:
df.tail() # to show the tail of my dataset

In [ ]:
df.shape # to show the dimension of my dataset

In [ ]:
df.info() # to have informations about each column of my dataset

## Description of the dataset

This dataset contains 2600 raw advertising records from a simulated Google Ads campaign designed to promote data analytics courses and services. It mirrors the reality of marketing data exports, including typos, inconsistent formats, missing values, and various irregularities. The 13 available variables such as Ad_ID, Campaign_Name, Clicks, Impressions, Cost, Leads, Conversions, Conversion Rate, Sale_Amount, Ad_Date, Location, Device, and Keyword cover the full performance lifecycle of an ad, from its delivery to its commercial outcomes, making this dataset an ideal foundation for data cleaning, exploratory analysis, and marketing performance evaluation.

The ‘Cost’ and ‘Sale_Amount’ columns need to be converted to float, and the ‘Ad_Date’ column needs to be converted to a proper date format.

In [ ]:

df['Cost'] = df['Cost'].str.replace('$', '', regex=False).astype(float)
df['Sale_Amount'] = df['Sale_Amount'].str.replace('$', '', regex=False).astype(float)
df['Ad_Date'] = pd.to_datetime(df['Ad_Date'], errors='coerce')

In [ ]:
df = df.sort_values(by='Ad_Date', ascending= True) # to sort the dates from the oldest to the newest

In [ ]:
df.head()

In [ ]:
df.tail()

Data cleaning


In [ ]:
# Let's drop the column "Ad_Date" because it is not necessary in this study
df.drop('Ad_Date', axis=1, inplace=True)

In [ ]:
df.shape

In [ ]:
df.describe() # to show some descriptive statistics

In [ ]:
df.isna().sum() # to check if there is any null value

In [ ]:
df.duplicated().sum() # to check if there is any duplicated value

Let's use KNN imputation to deal with missing values

In [ ]:
from sklearn.impute import KNNImputer
cols = ['Clicks','Impressions', 'Cost', 'Leads', 'Conversions','Conversion Rate','Sale_Amount']
imputer = KNNImputer(n_neighbors=5)
df_imputed = imputer.fit_transform(df[cols])

In [ ]:
df_imputed = pd.DataFrame(df_imputed, columns=cols)
df_imputed.head()

In [ ]:
df_imputed.isna().sum()

# Exploratory Data Analysis (EDA)

In [ ]:
df_imputed.columns

## Histograms of key variables

In [ ]:

sns.histplot(df_imputed['Clicks'], kde= True)
plt.title('Histogram of Clicks')
plt.xlabel('Clicks')
plt.ylabel('Frequency')
plt.show()

In [ ]:

sns.histplot(df_imputed['Impressions'], kde = True)
plt.title('Histogram of Impressions')
plt.xlabel('Impressions')
plt.ylabel('Frequency')
plt.show()

In [ ]:

sns.histplot(df_imputed['Cost'], kde = True)
plt.title('Histogram of Cost')
plt.xlabel('Cost')
plt.ylabel('Frequency')
plt.show()

In [ ]:
sns.histplot(df_imputed['Sale_Amount'], kde = True)
plt.title('Histogram of Sale_Amount')
plt.xlabel('Sale_Amount')
plt.ylabel('Frequency')
plt.show()

## Boxplot of key variables

In [ ]:
sns.boxplot(df_imputed['Clicks'])
plt.title('Boxplot of Clicks')
plt.xlabel('Clicks')
plt.show()

In [ ]:
sns.boxplot(df_imputed['Impressions'])
plt.title('Boxplot of Impressions')
plt.xlabel('Impressions')
plt.show()

In [ ]:
sns.boxplot(df_imputed['Cost'])
plt.title('Boxplot of Cost')
plt.xlabel('Cost')
plt.show()

In [ ]:
sns.boxplot(df_imputed['Sale_Amount'])
plt.title('Boxplot of Sale_Amount')
plt.xlabel('Sale_Amount')
plt.show()

## Let's deal with outliers

In [ ]:
from numpy._core.defchararray import upper
Q1 = df_imputed['Sale_Amount'].quantile(0.25)
Q3 = df_imputed['Sale_Amount'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR
outliers = df_imputed[(df_imputed['Sale_Amount'] < lower_bound) | (df_imputed['Sale_Amount'] > upper_bound)]
len(outliers)

Based on the boxplots generated for each variable and the outlier detection test performed on the ‘Sale_Amount’ variable, we can conclude that there are no outliers in any of the key variables

## Standardization of key variables

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
colms = ['Clicks','Impressions', 'Cost','Sale_Amount']
df_imputed[colms] = scaler.fit_transform(df_imputed[colms])

## Heatmap for key variables

In [ ]:
sns.heatmap(df_imputed[['Sale_Amount','Cost','Impressions', 'Clicks']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation between key variables')
plt.show()

## Scatter plot between Sale_Amount and other variables

In [ ]:
sns.scatterplot(x=df_imputed['Cost'], y=df_imputed['Sale_Amount'])
plt.title('Scatter plot between Sale_Amount and Cost')
plt.xlabel('Cost')
plt.ylabel('Sale_Amount')
plt.show()

In [ ]:
sns.scatterplot(x=df_imputed['Impressions'], y=df_imputed['Sale_Amount'])
plt.title('Scatter plot between Sale_Amount and Impressions')
plt.xlabel('Impressions')
plt.ylabel('Sale_Amount')
plt.show()

In [ ]:
sns.scatterplot(x=df_imputed['Clicks'], y=df_imputed['Sale_Amount'])
plt.title('Scatter plot between Sale_Amount and Clicks')
plt.xlabel('Clicks')
plt.ylabel('Sale_Amount')
plt.show()

# Building Linear Model

Let's split data into training and test sets to prepare for model training and evaluation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
y = df_imputed['Sale_Amount']
x = df_imputed.drop('Sale_Amount', axis=1)

In [ ]:
# split data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [ ]:
# create and train model
model = LinearRegression()
model.fit(x_train, y_train)

In [ ]:
# Make prediction
y_pred = model.predict(x_test)

In [ ]:
# Evaluation
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [ ]:
print('Mean Squared Error:', mse)
print('R-squared:', r2)

# Report

The model’s performance is very poor, which is expected because the model is misspecified. The relationship between the target variable and the feature variables is not linear. As a result, a linear model is not appropriate for capturing the underlying pattern in the data.

To address this issue, it would be necessary to consider alternative models that are better suited to handling non linear relationships between the target variable and the features.

Moreover, the dataset used is fundamentally a time series. Therefore, it would be more appropriate to explore time series models such as AR, MA, or ARIMA in order to better capture the true relationship and temporal dependencies between the variables.

# 1.3 Decision Tree Implementation

## 1.3.1 Decision Tree for regression : Google Ads campaign Dataset

In [ ]:
from sklearn.tree import DecisionTreeRegressor

In [ ]:
# Load the Google Ads Campaign Dataset : Here, it's the previous df_imputed
df_imputed.head()

In [ ]:
df_imputed.tail()

In [ ]:
# Split the data
X = df_imputed.drop('Sale_Amount', axis=1)
Y = df_imputed['Sale_Amount']
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42)

In [ ]:
# Create and Train the decision tree regressor
regressor = DecisionTreeRegressor(max_depth = 5, random_state = 42)
regressor.fit(X_train, Y_train)

In [ ]:
# Make predictions

Y_pred = regressor.predict(X_test)

In [ ]:
# Calculate the mean squared error
mse = mean_squared_error(Y_test, Y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(Y_test, Y_pred)

In [ ]:
print('Mean Squared Error:', mse)
print('Root Mean Squared Error:', rmse)
print('R-squared:', r2)

# Report
Once again, the model’s performance is poor when using a Decision Tree. This outcome is not surprising, as the dataset is fundamentally a time series. The correlation matrix already indicated that there is virtually no correlation between the target variable and the feature variables. In other words, there is no clear linear relationship between them. Therefore, it would be more appropriate to consider time series specific models such as AR, MA, or ARIMA, which are better suited to capturing the underlying temporal dynamics and relationships present in the data.



## 1.3.2 Decision Tree for Classification : Iris Dataset

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Load Iris dataset
iris = load_iris()
x, y = iris.data, iris.target

In [ ]:
df = pd.concat([pd.DataFrame(x, columns=iris.feature_names), pd.Series(y, name='target')], axis=1)
df.head()

In [ ]:
df.tail()

In [ ]:
# Split the data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, random_state = 42)

In [ ]:
# Create and train the the decision tree classifier
classifier = DecisionTreeClassifier(max_depth=3, random_state=42)
classifier.fit(x_train, y_train)

In [ ]:
# Make prediction
y_pred = classifier.predict(x_test)

In [ ]:
# calculate accuracy and print classification report
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=iris.target_names)

In [ ]:
print(accuracy)

In [ ]:
print(report)

# Report
Based on the results obtained with the DecisionTreeClassifier on the Iris dataset, the model reaches an accuracy of 1.0, indicating that it correctly classifies all instances in the test set. This reflects the well structured nature of the Iris dataset, where the species are clearly distinguishable using the available features. The decision tree is able to capture these distinctions effectively, leading to perfect classification performance on this task.